In [1]:
from pathlib import Path
import pandas as pd

raw = Path("../data/raw")

for path in sorted(raw.glob("*.xlsx")):
    print(path.name)
    for sheet in pd.ExcelFile(path).sheet_names:
        print("   -", sheet)
    print()

Global-Energy-Ownership-Tracker-November-2026-V1.xlsx
   - About
   - All Entities
   - Entity Ownership
   - Asset Ownership
   - Coal Plant Ownership
   - Gas Plant Ownership
   - Bioenergy Power Ownership
   - Coal Mine Ownership
   - Iron Mine Ownership
   - Gas Pipeline Ownership
   - Oil & NGL Pipeline Ownership
   - Steel Plant Ownership
   - Cement and Concrete Ownership
   - LNG Terminal Ownership
   - Chemicals Ownership

Iron unit data - Global Iron and Steel Tracker - September 2026 (V1).xlsx
   - About
   - Metadata
   - Blast furnaces
   - Direct reduced iron furnaces
   - Blast furnace relinings

Plant-level data - Global Iron and Steel Tracker - September 2026 (V1).xlsx
   - About
   - Metadata
   - Plant data
   - Plant capacities and status
   - Plant production

Steel unit data - Global Iron and Steel Tracker - September 2026 (V1).xlsx
   - About
   - Metadata
   - Electric arc furnaces
   - Basic oxygen furnaces
   - Induction furnaces
   - Open hearth furnaces



In [3]:
plant_file = raw / "Plant-level data - Global Iron and Steel Tracker - September 2026 (V1).xlsx"

for sheet in ["Plant data", "Plant capacities and status", "Plant production"]:
    df = pd.read_excel(plant_file, sheet_name=sheet)
    print("=====", sheet, "=====")
    print("rows, columns:", df.shape)
    print(list(df.columns))
    print()

===== Plant data =====
rows, columns: (1293, 44)
['GEM plant ID', 'Plant name (English)', 'Plant name (other language)', 'Other plant names (English)', 'Other plant names (other language)', 'Owner', 'Owner (other language)', 'Owner GEM entity ID', 'Owner PermID', 'SOE status', 'Parent (English)', 'Parent GEM entity ID', 'Parent PermID', 'Location address', 'Location address (other language)', 'Municipality', 'Subnational unit', 'Country/area', 'Region', 'Coordinates', 'Coordinate accuracy', 'GEM wiki page', 'Plant age', 'Announced date', 'Construction date', 'Start date', 'Pre-retirement announcement date', 'Idled date', 'Retired date', 'Ferronickel capacity (ttpa)', 'Sinter plant capacity (ttpa)', 'Coking plant capacity (ttpa)', 'Pelletizing plant capacity (ttpa)', 'Category steel product', 'Steel products', 'Steel sector end users', 'Workforce size', 'ISO 14001', 'ISO 50001', 'ResponsibleSteel certification', 'Main production equipment', 'Power source', 'Iron ore source', 'Met coal s

In [4]:
plants = pd.read_excel(plant_file, sheet_name="Plant data")
caps = pd.read_excel(plant_file, sheet_name="Plant capacities and status")
prod = pd.read_excel(plant_file, sheet_name="Plant production")

print("Plant IDs unique in Plant data?", plants["GEM plant ID"].is_unique)
print()
print("Status values in capacities:")
print(caps["Status"].value_counts())
print()
print("Plants with more than one capacity row:", (caps["GEM plant ID"].value_counts() > 1).sum())
print()
print("Production types:")
print(prod["Type of production"].value_counts())

Plant IDs unique in Plant data? True

Status values in capacities:
Status
operating                    935
announced                    288
retired                      233
construction                 140
operating pre-retirement      97
cancelled                     76
mothballed                    72
mothballed pre-retirement      7
Name: count, dtype: int64

Plants with more than one capacity row: 392

Production types:
Type of production
Crude steel production (ttpa)            431
EAF steel production (ttpa)              264
Iron production (ttpa)                   193
BOF steel production (ttpa)              173
BF production (ttpa)                     157
Other/Unknown steel production (ttpa)     80
DRI production (ttpa)                     65
Other/Unknown iron production (ttpa)      28
OHF steel production (ttpa)                2
Name: count, dtype: int64


In [5]:
import duckdb
import re

def clean(col):
    col = str(col).strip().lower()
    if col.isdigit():                    # year columns: 2019 -> y2019
        return "y" + col
    col = re.sub(r"[^a-z0-9]+", "_", col)
    return col.strip("_")

for df in (plants, caps, prod):
    df.columns = [clean(c) for c in df.columns]

con = duckdb.connect("../db/steel.duckdb")
con.execute("CREATE OR REPLACE TABLE raw_plants AS SELECT * FROM plants")
con.execute("CREATE OR REPLACE TABLE raw_capacities AS SELECT * FROM caps")
con.execute("CREATE OR REPLACE TABLE raw_production AS SELECT * FROM prod")

print(con.execute("""
    SELECT 'raw_plants' AS tbl, COUNT(*) AS n FROM raw_plants
    UNION ALL SELECT 'raw_capacities', COUNT(*) FROM raw_capacities
    UNION ALL SELECT 'raw_production', COUNT(*) FROM raw_production
""").df())

print(con.execute("DESCRIBE raw_capacities").df()[["column_name", "column_type"]])

              tbl     n
0      raw_plants  1293
1  raw_capacities  1848
2  raw_production  1393
                              column_name column_type
0                            gem_plant_id     VARCHAR
1                      plant_name_english     VARCHAR
2               plant_name_other_language     VARCHAR
3                            country_area     VARCHAR
4               main_production_equipment     VARCHAR
5                                  status     VARCHAR
6                              start_date     VARCHAR
7       nominal_crude_steel_capacity_ttpa     VARCHAR
8         nominal_bof_steel_capacity_ttpa     VARCHAR
9         nominal_eaf_steel_capacity_ttpa     VARCHAR
10         nominal_if_steel_capacity_ttpa     VARCHAR
11  other_unspecified_steel_capacity_ttpa      DOUBLE
12             nominal_iron_capacity_ttpa     VARCHAR
13               nominal_bf_capacity_ttpa     VARCHAR
14              nominal_dri_capacity_ttpa     VARCHAR
15   other_unspecified_iron_capacity_ttp

In [7]:
print(con.execute("""
    WITH as_text AS (
        SELECT gem_plant_id, COLUMNS('capacity_ttpa$')::VARCHAR
        FROM raw_capacities
    ),
    long AS (
        UNPIVOT as_text
        ON COLUMNS('capacity_ttpa$')
        INTO NAME capacity_col VALUE raw_value
    )
    SELECT capacity_col, raw_value, COUNT(*) AS n
    FROM long
    WHERE TRY_CAST(raw_value AS DOUBLE) IS NULL
    GROUP BY ALL
    ORDER BY capacity_col, n DESC
""").df().to_string())

                           capacity_col raw_value   n
0              nominal_bf_capacity_ttpa        >0   6
1       nominal_bof_steel_capacity_ttpa        >0   1
2     nominal_crude_steel_capacity_ttpa        >0  16
3             nominal_dri_capacity_ttpa        >0  18
4       nominal_eaf_steel_capacity_ttpa        >0  10
5        nominal_if_steel_capacity_ttpa        >0   6
6            nominal_iron_capacity_ttpa        >0  34
7  other_unspecified_iron_capacity_ttpa        >0  13


In [8]:
con.execute("""
    CREATE OR REPLACE VIEW capacities AS
    SELECT
        gem_plant_id,
        plant_name_english AS plant_name,
        country_area AS country,
        status,
        status IN ('operating', 'operating pre-retirement') AS is_current,
        TRY_CAST(COLUMNS('capacity_ttpa$') AS DOUBLE),
        (nominal_crude_steel_capacity_ttpa = '>0'
         OR nominal_eaf_steel_capacity_ttpa = '>0'
         OR nominal_bof_steel_capacity_ttpa = '>0') AS steel_capacity_unknown
    FROM raw_capacities
""")

print(con.execute("""
    SELECT is_current,
           COUNT(*) AS rows,
           SUM(steel_capacity_unknown::INT) AS rows_unknown,
           ROUND(SUM(nominal_crude_steel_capacity_ttpa) / 1000) AS crude_mtpa,
           ROUND(SUM(nominal_eaf_steel_capacity_ttpa) / 1000) AS eaf_mtpa,
           ROUND(SUM(nominal_bof_steel_capacity_ttpa) / 1000) AS bof_mtpa
    FROM capacities
    GROUP BY is_current
""").df())

   is_current  rows  rows_unknown  crude_mtpa  eaf_mtpa  bof_mtpa
0       False   816          12.0      1444.0     505.0     775.0
1        True  1032           4.0      2228.0     720.0    1459.0


In [9]:
print(con.execute("""
    SELECT
        COUNT(*) AS current_rows,
        SUM(CASE WHEN ABS(diff) <= 1 THEN 1 ELSE 0 END) AS matches,
        SUM(CASE WHEN ABS(diff) > 1 THEN 1 ELSE 0 END) AS mismatches,
        SUM(CASE WHEN diff IS NULL THEN 1 ELSE 0 END) AS cannot_check
    FROM (
        SELECT nominal_crude_steel_capacity_ttpa
             - (COALESCE(nominal_bof_steel_capacity_ttpa, 0)
              + COALESCE(nominal_eaf_steel_capacity_ttpa, 0)
              + COALESCE(nominal_if_steel_capacity_ttpa, 0)
              + COALESCE(other_unspecified_steel_capacity_ttpa, 0)) AS diff
        FROM capacities
        WHERE is_current
    )
""").df())

   current_rows  matches  mismatches  cannot_check
0          1032    947.0         0.0          85.0


In [10]:
print(con.execute("""
    SELECT
        CASE
            WHEN steel_capacity_unknown THEN 'steel capacity >0 (unknown size)'
            WHEN nominal_iron_capacity_ttpa IS NOT NULL
                 OR nominal_bf_capacity_ttpa IS NOT NULL
                 OR nominal_dri_capacity_ttpa IS NOT NULL THEN 'iron only'
            ELSE 'no capacity at all'
        END AS reason,
        COUNT(*) AS rows
    FROM capacities
    WHERE is_current AND nominal_crude_steel_capacity_ttpa IS NULL
    GROUP BY reason
    ORDER BY rows DESC
""").df())

                             reason  rows
0                         iron only    81
1  steel capacity >0 (unknown size)     4


In [11]:
print(con.execute("""
    SELECT owner, parent_english, parent_gem_entity_id
    FROM raw_plants
    WHERE parent_english LIKE '%;%' OR parent_english LIKE '%[%'
    LIMIT 10
""").df().to_string())

print()
print(con.execute("""
    SELECT
        COUNT(*) AS plants,
        SUM(CASE WHEN parent_english IS NULL THEN 1 ELSE 0 END) AS no_parent,
        SUM(CASE WHEN parent_english LIKE '%;%' THEN 1 ELSE 0 END) AS multiple_parents
    FROM raw_plants
""").df())

                                       owner                                                                       parent_english                          parent_gem_entity_id
0                         ABA Çelik Demir LŞ                                                          ABA Çelik Demir LŞ [100.0%]                        E100000131190 [100.0%]
1                             Abba Steel Ltd                                                              Abba Steel Ltd [100.0%]                        E100001012072 [100.0%]
2  Abinski Elektrometallurgicheski Zavod LLC                                                              Novostal-M LLC [100.0%]                        E100000131027 [100.0%]
3                    Acciaierie d'italia SpA                                                     Acciaierie d'italia SpA [100.0%]                        E100001010116 [100.0%]
4                      Acciaierie Venete SpA                                                       Acciaierie Venete SpA

In [12]:
con.execute(r"""
    CREATE OR REPLACE VIEW plant_parents AS
    WITH split AS (
        SELECT gem_plant_id,
               unnest(string_split(parent_english, '; ')) AS parent_txt,
               unnest(string_split(parent_gem_entity_id, '; ')) AS id_txt
        FROM raw_plants
    )
    SELECT gem_plant_id,
           trim(regexp_extract(parent_txt, '^(.*) \[', 1)) AS parent_name,
           regexp_extract(id_txt, '^(\S+)', 1) AS parent_id,
           TRY_CAST(regexp_extract(parent_txt, '\[([0-9.]+)%\]', 1) AS DOUBLE) / 100 AS share
    FROM split
""")

print(con.execute("SELECT * FROM plant_parents WHERE gem_plant_id IN "
                  "(SELECT gem_plant_id FROM plant_parents GROUP BY 1 HAVING COUNT(*) > 1) LIMIT 6").df())

print()
print(con.execute("""
    SELECT
        COUNT(*) AS parent_rows,
        SUM(CASE WHEN share IS NULL THEN 1 ELSE 0 END) AS share_missing,
        SUM(CASE WHEN parent_name = '' THEN 1 ELSE 0 END) AS name_missing
    FROM plant_parents
""").df())

print()
print(con.execute("""
    SELECT ROUND(total, 2) AS share_total, COUNT(*) AS plants
    FROM (SELECT gem_plant_id, SUM(share) AS total FROM plant_parents GROUP BY 1)
    GROUP BY 1 ORDER BY plants DESC LIMIT 8
""").df())

    gem_plant_id                                        parent_name  \
0  P100000121029                                        Acerinox SA   
1  P100000121029  Industrial Development Corporation of South Af...   
2  P100001025815                                  natural person(s)   
3  P100001025815                               small shareholder(s)   
4  P100001025815                         Mahanada Suppliers Pvt Ltd   
5  P100001025815                            Sunrwoth Stocks Pvt Ltd   

       parent_id  share  
0  E100000130743  0.760  
1  E100001000576  0.240  
2  E100000123261  0.361  
3  E100001015587  0.288  
4  E100001016502  0.226  
5  E100001016503  0.125  

   parent_rows  share_missing  name_missing
0         2219          326.0         326.0

   share_total  plants
0         1.00    1055
1          NaN     171
2         0.98       9
3         2.00       6
4         0.11       4
5         0.93       3
6         1.01       3
7         0.95       3


In [13]:
print("--- Entries the pattern could not read ---")
print(con.execute(r"""
    SELECT parent_txt, COUNT(*) AS n
    FROM (SELECT unnest(string_split(parent_english, '; ')) AS parent_txt FROM raw_plants)
    WHERE parent_txt NOT SIMILAR TO '.* \[[0-9.]+%\]'
    GROUP BY 1 ORDER BY n DESC LIMIT 10
""").df().to_string())

print()
print("--- Plants whose shares add up to 200% ---")
print(con.execute("""
    SELECT p.gem_plant_id, r.parent_english, r.parent_gem_entity_id
    FROM (SELECT gem_plant_id FROM plant_parents GROUP BY 1 HAVING ROUND(SUM(share), 2) = 2) p
    JOIN raw_plants r USING (gem_plant_id)
    LIMIT 3
""").df().to_string())

--- Entries the pattern could not read ---
                                                                parent_txt   n
0                                                     small shareholder(s)  13
1  Iranian Mines & Mining Industries Development & Renovation Organization  10
2                                                 Riva Forni Elettrici SpA  10
3                                                         GFG Alliance Ltd   9
4                                                                  unknown   6
5                                                         Deutsche Bank AG   5
6                                             Strategic Value Partners LLC   5
7                                                    Attestor Services Ltd   5
8                                           GoldenTree Asset Management LP   5
9                                            Anchorage Capital Advisors LP   5

--- Plants whose shares add up to 200% ---
    gem_plant_id                            

In [14]:
con.execute(r"""
    CREATE OR REPLACE VIEW plant_parents AS
    WITH split AS (
        SELECT gem_plant_id,
               unnest(string_split(parent_english, '; ')) AS parent_txt,
               unnest(string_split(parent_gem_entity_id, '; ')) AS id_txt
        FROM raw_plants
    ),
    parsed AS (
        SELECT gem_plant_id,
               trim(regexp_replace(parent_txt, '\s*\[[0-9.]+%\]$', '')) AS parent_name,
               regexp_extract(id_txt, '^(\S+)', 1) AS parent_id,
               TRY_CAST(regexp_extract(parent_txt, '\[([0-9.]+)%\]', 1) AS DOUBLE) / 100 AS share_reported
        FROM split
    ),
    filled AS (
        SELECT *,
               CASE WHEN share_reported IS NOT NULL THEN share_reported
                    ELSE GREATEST(1 - COALESCE(SUM(share_reported) OVER w, 0), 0)
                         / COUNT(*) FILTER (WHERE share_reported IS NULL) OVER w
               END AS share_filled
        FROM parsed
        WINDOW w AS (PARTITION BY gem_plant_id)
    )
    SELECT gem_plant_id, parent_name, parent_id, share_reported,
           CASE WHEN SUM(share_filled) OVER w > 1.01
                THEN share_filled / SUM(share_filled) OVER w
                ELSE share_filled END AS share,
           CASE WHEN SUM(share_filled) OVER w > 1.01 THEN 'normalised'
                WHEN share_reported IS NULL THEN 'estimated'
                ELSE 'reported' END AS share_method
    FROM filled
    WINDOW w AS (PARTITION BY gem_plant_id)
""")

print(con.execute("""
    SELECT share_method, COUNT(*) AS rows FROM plant_parents GROUP BY 1
""").df())
print()
print(con.execute("""
    SELECT ROUND(total, 2) AS share_total, COUNT(*) AS plants
    FROM (SELECT gem_plant_id, SUM(share) AS total FROM plant_parents GROUP BY 1)
    GROUP BY 1 ORDER BY plants DESC LIMIT 8
""").df())

  share_method  rows
0   normalised    21
1     reported  1872
2    estimated   326

   share_total  plants
0         1.00    1264
1         0.98       9
2         0.95       3
3         0.99       3
4         0.93       2
5         1.01       2
6         0.49       1
7         0.35       1


In [15]:
print(con.execute("""
    WITH company_cap AS (
        SELECT
            pp.parent_id,
            ANY_VALUE(pp.parent_name) AS company,
            COUNT(DISTINCT c.gem_plant_id) AS plants,
            SUM(c.nominal_crude_steel_capacity_ttpa * pp.share) / 1000 AS crude_mtpa,
            SUM(c.nominal_eaf_steel_capacity_ttpa * pp.share) / 1000 AS eaf_mtpa
        FROM capacities c
        JOIN plant_parents pp USING (gem_plant_id)
        WHERE c.is_current
        GROUP BY pp.parent_id
    )
    SELECT
        RANK() OVER (ORDER BY crude_mtpa DESC) AS rank,
        company,
        plants,
        ROUND(crude_mtpa, 1) AS crude_mtpa,
        ROUND(100 * COALESCE(eaf_mtpa, 0) / crude_mtpa, 0) AS eaf_share_pct
    FROM company_cap
    WHERE crude_mtpa IS NOT NULL
    ORDER BY rank
    LIMIT 20
""").df().to_string(index=False))

 rank                            company  plants  crude_mtpa  eaf_share_pct
    1                  Nippon Steel Corp      19        74.2           18.0
    2                   ArcelorMittal SA      29        67.4           38.0
    3        Baoshan Iron & Steel Co Ltd       4        54.5           10.0
    4                 POSCO Holdings Inc       4        44.3            5.0
    5                            unknown     117        38.7           23.0
    6                      JSW Steel Ltd       7        34.9           19.0
    7                     Tata Steel Ltd       7        34.0            7.0
    8                   JFE Holdings Inc       8        33.5           10.0
    9                  natural person(s)      50        33.2           29.0
   10                Angang Steel Co Ltd       3        30.3            0.0
   11 Beijing Jianlong Investment Co Ltd      14        28.2            4.0
   12                         Nucor Corp      20        26.8          100.0
   13       

In [16]:
own_file = raw / "Global-Energy-Ownership-Tracker-November-2026-V1.xlsx"
ent_own = pd.read_excel(own_file, sheet_name="Entity Ownership")

print("rows, columns:", ent_own.shape)
print(list(ent_own.columns))
print()
print(ent_own.head(5).to_string())

rows, columns: (29322, 7)
['Subject Entity ID', 'Subject Entity Name', 'Interested Party ID', 'Interested Party Name', '% Share of Ownership', 'Data Source URL', 'Share Imputed?']

  Subject Entity ID    Subject Entity Name Interested Party ID   Interested Party Name  % Share of Ownership                                                                                                                                                                                                                  Data Source URL            Share Imputed?
0     E100000000003              Sonatrach       E100001000008   Government of Algeria                 100.0                  https://www.thenationalnews.com/business/energy/2022/08/25/algerias-sonatrach-makes-new-oil-discovery-in-adrar-province/,https://www.worldbenchmarkingalliance.org/publication/oil-and-gas/companies/sonatrach/  known value, not imputed
1     E100000000004               Sonangol       E100001012407    Government of Angola             

In [17]:
ent_own.columns = [clean(c) for c in ent_own.columns]
con.execute("CREATE OR REPLACE TABLE raw_entity_ownership AS SELECT * FROM ent_own")
print(list(ent_own.columns))
print()

print(con.execute("""
    WITH RECURSIVE chain AS (
        SELECT subject_entity_id, subject_entity_name,
               interested_party_id, interested_party_name,
               share_of_ownership, 1 AS level
        FROM raw_entity_ownership
        WHERE subject_entity_name = 'Baoshan Iron & Steel Co Ltd'
        UNION ALL
        SELECT o.subject_entity_id, o.subject_entity_name,
               o.interested_party_id, o.interested_party_name,
               o.share_of_ownership, c.level + 1
        FROM raw_entity_ownership o
        JOIN chain c ON o.subject_entity_id = c.interested_party_id
        WHERE c.level < 6
    )
    SELECT level, subject_entity_name, interested_party_name, share_of_ownership
    FROM chain ORDER BY level, share_of_ownership DESC
""").df().to_string(index=False))

['subject_entity_id', 'subject_entity_name', 'interested_party_id', 'interested_party_name', 'share_of_ownership', 'data_source_url', 'share_imputed']

Empty DataFrame
Columns: [level, subject_entity_name, interested_party_name, share_of_ownership]
Index: []


In [18]:
print(con.execute("""
    SELECT DISTINCT parent_id, parent_name
    FROM plant_parents WHERE parent_name LIKE 'Baoshan%'
""").df())
print()

print(con.execute("""
    WITH RECURSIVE chain AS (
        SELECT subject_entity_id, subject_entity_name,
               interested_party_id, interested_party_name,
               share_of_ownership, 1 AS level
        FROM raw_entity_ownership
        WHERE subject_entity_id IN (
            SELECT parent_id FROM plant_parents WHERE parent_name LIKE 'Baoshan%')
        UNION ALL
        SELECT o.subject_entity_id, o.subject_entity_name,
               o.interested_party_id, o.interested_party_name,
               o.share_of_ownership, c.level + 1
        FROM raw_entity_ownership o
        JOIN chain c ON o.subject_entity_id = c.interested_party_id
        WHERE c.level < 6
    )
    SELECT level, subject_entity_name, interested_party_name, share_of_ownership
    FROM chain ORDER BY level, share_of_ownership DESC
""").df().to_string(index=False))

       parent_id                  parent_name
0  E100000122571  Baoshan Iron & Steel Co Ltd

Empty DataFrame
Columns: [level, subject_entity_name, interested_party_name, share_of_ownership]
Index: []


In [19]:
bao_id = "E100000122571"

print("Baoshan ID as owned company:",
      con.execute(f"SELECT COUNT(*) FROM raw_entity_ownership WHERE subject_entity_id = '{bao_id}'").fetchone()[0])
print("Baoshan ID as owner:",
      con.execute(f"SELECT COUNT(*) FROM raw_entity_ownership WHERE interested_party_id = '{bao_id}'").fetchone()[0])
print()

print(con.execute("""
    SELECT DISTINCT subject_entity_id, subject_entity_name, interested_party_name, share_of_ownership
    FROM raw_entity_ownership
    WHERE subject_entity_name ILIKE '%baoshan%' OR subject_entity_name ILIKE '%baowu%'
       OR interested_party_name ILIKE '%baowu%'
    LIMIT 10
""").df().to_string(index=False))

Baoshan ID as owned company: 0
Baoshan ID as owner: 3

subject_entity_id                           subject_entity_name                         interested_party_name  share_of_ownership
    E100000122269                  Yuanbaoshan Power Generation                  Guodian Inner Mongolia Power                97.9
    E100000123215                            Xinyu Iron & Steel                       China Baowu Steel Group                51.0
    E100000123228 Baosteel Group Guangdong Shaoguan Iron &STEEL                       China Baowu Steel Group                51.0
    E100000123229               Baowu Group Echeng Iron & Steel Baosteel Group Guangdong Shaoguan Iron &STEEL                51.0
    E100000123229               Baowu Group Echeng Iron & Steel                                  Wugang Group                23.8
    E100000123229               Baowu Group Echeng Iron & Steel                       China Baowu Steel Group                25.2
    E100000123274                  

In [21]:
print(con.execute("""
    WITH company_cap AS (
        SELECT pp.parent_id, ANY_VALUE(pp.parent_name) AS company,
               COUNT(DISTINCT c.gem_plant_id) AS plants,
               SUM(c.nominal_crude_steel_capacity_ttpa * pp.share) / 1000 AS crude_mtpa,
               SUM(c.nominal_eaf_steel_capacity_ttpa * pp.share) / 1000 AS eaf_mtpa
        FROM capacities c JOIN plant_parents pp USING (gem_plant_id)
        WHERE c.is_current
        GROUP BY pp.parent_id
    )
    SELECT company, parent_id, plants,
           ROUND(crude_mtpa, 1) AS crude_mtpa,
           ROUND(100 * COALESCE(eaf_mtpa, 0) / crude_mtpa, 0) AS eaf_share_pct
    FROM company_cap
    WHERE regexp_matches(company, '(?i)thyssenkrupp|salzgitter|arcelormittal|tata steel|ssab|voestalpine|nucor|posco')
    ORDER BY company
""").df().to_string(index=False))

                                 company     parent_id  plants  crude_mtpa  eaf_share_pct
ArcelorMittal Atlantique et Lorraine SAS E100001000487       1         6.8            0.0
                        ArcelorMittal SA E100000000687      29        67.4           38.0
          ArcelorMittal South Africa Ltd E100001000504       1         3.4            0.0
      ArcelorMittal Tubarão Comercial SA E100001000507       1         7.5            0.0
                              Nucor Corp E100001010181      20        26.8          100.0
                      POSCO Holdings Inc E100000000953       4        44.3            5.0
                                 SSAB AB E100000001651       5         8.9           28.0
                           Salzgitter AG E100001010315       3         7.5           13.0
               Tata Steel (Thailand) PCL E100001010159       2         1.4          100.0
            Tata Steel Long Products Ltd E100000121408       1         1.0            0.0
          

In [22]:
con.execute("""
    CREATE OR REPLACE TABLE company_map AS
    SELECT * FROM (VALUES
        ('E100000000687', 'ArcelorMittal'),
        ('E100001000487', 'ArcelorMittal'),
        ('E100001000504', 'ArcelorMittal'),
        ('E100001000507', 'ArcelorMittal'),
        ('E100001010181', 'Nucor'),
        ('E100000000953', 'POSCO'),
        ('E100000001651', 'SSAB'),
        ('E100001010315', 'Salzgitter'),
        ('E100001010018', 'Tata Steel'),
        ('E100001010159', 'Tata Steel'),
        ('E100000121408', 'Tata Steel'),
        ('E100000000031', 'voestalpine'),
        ('E100000004212', 'thyssenkrupp')
    ) AS t(parent_id, company)
""")

print(con.execute("""
    SELECT m.company,
           COUNT(DISTINCT c.gem_plant_id) AS plants,
           ROUND(SUM(c.nominal_crude_steel_capacity_ttpa * pp.share) / 1000, 1) AS crude_mtpa,
           ROUND(100 * SUM(COALESCE(c.nominal_eaf_steel_capacity_ttpa, 0) * pp.share)
                     / SUM(c.nominal_crude_steel_capacity_ttpa * pp.share), 0) AS eaf_share_pct
    FROM capacities c
    JOIN plant_parents pp USING (gem_plant_id)
    JOIN company_map m USING (parent_id)
    WHERE c.is_current
    GROUP BY m.company
    ORDER BY crude_mtpa DESC
""").df().to_string(index=False))

      company  plants  crude_mtpa  eaf_share_pct
ArcelorMittal      32        85.0           30.0
        POSCO       4        44.3            5.0
   Tata Steel       9        36.4           10.0
        Nucor      20        26.8          100.0
 thyssenkrupp       3        11.3            2.0
         SSAB       5         8.9           28.0
  voestalpine       2         7.6            0.0
   Salzgitter       3         7.5           13.0


In [23]:
con.close()